# Практика 1. Две витрины из одного источника

Полное решение для самостоятельного прохождения после занятия.

После `docker compose up -d` PostgreSQL уже содержит учебные данные. Сначала строим первую витрину в pandas, затем сохраняем исходные строки в Parquet в MinIO и строим вторую витрину запросом ClickHouse. В конце сравниваем обе витрины. Ноутбук запускается на компьютере, не в Docker.


## 1. Окружение и подключения

Если VS Code попросит перезапустить kernel после установки пакетов, сделайте это.


In [ ]:
%pip install -q "psycopg[binary]" boto3 clickhouse-connect pandas pyarrow


In [7]:
import io
from datetime import date

import boto3
from botocore.config import Config
import clickhouse_connect
import pandas as pd
import psycopg

PG = dict(host='localhost', port=5433, dbname='shop', user='student', password='studentpass')
CH = dict(host='localhost', port=8123, database='de_course', username='student', password='studentpass')
S3_ENDPOINT = 'http://localhost:9010'  # ноутбук → MinIO с компьютера
S3_USER = 'minioadmin'
S3_PASSWORD = 'minioadmin'
BUCKET = 'de-course'
KEY = f'raw/orders/load_date={date.today().isoformat()}/orders.parquet'
S3_URL_FOR_CLICKHOUSE = f'http://minio:9000/{BUCKET}/{KEY}'  # ClickHouse → MinIO внутри Docker


## 2. Общий источник — PostgreSQL

В DBeaver проверьте число заказов и распределение статусов. Ноутбук соединит заказы с кампаниями по `order_id`.


In [2]:
sql = '''
SELECT o.order_id, o.order_date::text AS order_date,
       o.status, o.amount_rub, a.campaign
FROM orders AS o
LEFT JOIN marketing_attribution AS a USING (order_id)
ORDER BY o.order_id
'''
with psycopg.connect(**PG) as conn:
    with conn.cursor() as cur:
        cur.execute(sql)
        columns = [column.name for column in cur.description]
        records = [dict(zip(columns, row)) for row in cur.fetchall()]
assert len(records) == 120
print('Прочитано заказов:', len(records))
display(pd.DataFrame(records).head())


Прочитано заказов: 120


,order_id,order_date,status,amount_rub,campaign
0,1001,2026-09-01,returned,500,spring_ads
1,1002,2026-09-02,pending,1200,spring_ads
2,1003,2026-09-03,paid,1900,spring_ads
3,1004,2026-09-04,paid,600,spring_ads
4,1005,2026-09-05,paid,1300,spring_ads


**Контрольная точка 1.** Почему общую сумму всех заказов нельзя назвать выручкой? **Ваш ответ:** …


## 3. Подход 1 — преобразовать в pandas, затем загрузить

Сводная таблица суммирует заказы по кампании и статусу. Оставляем `paid` и `returned`; столбец `pending` не берём. Для этой учебной выборки у каждой кампании есть оба нужных статуса. Преобразование происходит **до** загрузки в ClickHouse.


In [3]:
source_df = pd.DataFrame(records)
totals = source_df.pivot_table(
    index='campaign', columns='status', values='amount_rub', aggfunc='sum'
).reset_index()
pandas_mart = totals[['campaign', 'paid', 'returned']].rename(
    columns={'paid': 'paid_rub', 'returned': 'returned_rub'}
)
pandas_mart['net_rub'] = pandas_mart['paid_rub'] - pandas_mart['returned_rub']
pandas_rows_to_load = pandas_mart[['campaign', 'paid_rub', 'returned_rub', 'net_rub']].values.tolist()
display(pandas_mart)


status,campaign,paid_rub,returned_rub,net_rub
0,email,32800,4300,28500
1,loyalty,37600,3400,34200
2,search,34000,5200,28800
3,spring_ads,34400,2500,31900


In [5]:
ch = clickhouse_connect.get_client(**CH)
ch.command('''
CREATE TABLE IF NOT EXISTS de_course.campaign_revenue_pandas (
    campaign String, paid_rub Int64, returned_rub Int64, net_rub Int64
) ENGINE = MergeTree ORDER BY campaign
''')
ch.command('TRUNCATE TABLE de_course.campaign_revenue_pandas')
ch.insert(
    'de_course.campaign_revenue_pandas', pandas_rows_to_load,
    column_names=['campaign', 'paid_rub', 'returned_rub', 'net_rub'],
)
pandas_rows = ch.query('''
SELECT campaign, paid_rub, returned_rub, net_rub
FROM de_course.campaign_revenue_pandas ORDER BY campaign
''').result_rows
print('Подход 1:', pandas_rows)


Подход 1: [('email', 32800, 4300, 28500), ('loyalty', 37600, 3400, 34200), ('search', 34000, 5200, 28800), ('spring_ads', 34400, 2500, 31900)]


**Контрольная точка 2.** Где именно выполнено преобразование данных в этой ветке? **Ваш ответ:** …


## 4. Подход 2 — загрузить исходные строки в Parquet

DataFrame ниже используется только для сериализации в Parquet. Здесь ещё **нет** группировки или расчёта выручки.


In [8]:
s3 = boto3.client(
    's3', endpoint_url=S3_ENDPOINT,
    aws_access_key_id=S3_USER, aws_secret_access_key=S3_PASSWORD,
    region_name='us-east-1',
    config=Config(signature_version='s3v4', s3={'addressing_style': 'path'}),
)
existing = {item['Name'] for item in s3.list_buckets()['Buckets']}
if BUCKET not in existing:
    s3.create_bucket(Bucket=BUCKET)
raw_df = pd.DataFrame(records)
buffer = io.BytesIO()
raw_df.to_parquet(buffer, engine='pyarrow', index=False)
s3.put_object(Bucket=BUCKET, Key=KEY, Body=buffer.getvalue(), ContentType='application/octet-stream',
              Metadata={'source': 'postgres', 'layer': 'raw'})
print(f's3://{BUCKET}/{KEY}', '-', buffer.tell(), 'байт')


s3://de-course/raw/orders/load_date=2026-11-04/orders.parquet - 4293 байт


In [9]:
response = s3.head_object(Bucket=BUCKET, Key=KEY)
print('Объект найден; размер:', response['ContentLength'], 'байт')
raw_count_in_ch = ch.query(f"""
SELECT count() FROM s3('{S3_URL_FOR_CLICKHOUSE}', '{S3_USER}', '{S3_PASSWORD}', 'Parquet')
""").result_rows[0][0]
assert raw_count_in_ch == len(records)
print('ClickHouse прочитал из Parquet:', raw_count_in_ch, 'строк')


Объект найден; размер: 4293 байт
ClickHouse прочитал из Parquet: 120 строк


**Контрольная точка 3.** Покажите объект в MinIO Console. Почему ноутбук использует `localhost:9000`, а ClickHouse — `minio:9000`? **Ваш ответ:** …


## 5. Подход 2 — преобразовать в ClickHouse

ClickHouse читает исходный Parquet из MinIO и сам считает агрегат в SQL. Витрина записывается в другую таблицу.


In [10]:
ch.command('''
CREATE TABLE IF NOT EXISTS de_course.campaign_revenue_s3 (
    campaign String, paid_rub Int64, returned_rub Int64, net_rub Int64
) ENGINE = MergeTree ORDER BY campaign
''')
ch.command('TRUNCATE TABLE de_course.campaign_revenue_s3')
s3_sql = f"""
INSERT INTO de_course.campaign_revenue_s3
SELECT campaign, paid_rub, returned_rub, paid_rub - returned_rub AS net_rub
FROM (
    SELECT campaign,
           sumIf(toInt64(amount_rub), status = 'paid') AS paid_rub,
           sumIf(toInt64(amount_rub), status = 'returned') AS returned_rub
    FROM s3('{S3_URL_FOR_CLICKHOUSE}', '{S3_USER}', '{S3_PASSWORD}', 'Parquet')
    GROUP BY campaign
)
"""
ch.command(s3_sql)
s3_rows = ch.query('''
SELECT campaign, paid_rub, returned_rub, net_rub
FROM de_course.campaign_revenue_s3 ORDER BY campaign
''').result_rows
print('Подход 2:', s3_rows)


Подход 2: [('email', 32800, 4300, 28500), ('loyalty', 37600, 3400, 34200), ('search', 34000, 5200, 28800), ('spring_ads', 34400, 2500, 31900)]


## 6. Сравнить две витрины

Повторно выполните целиком ячейки загрузки обоих подходов. После повторов результат должен остаться прежним.


In [11]:
assert pandas_rows == s3_rows, 'Витрины различаются: сравните правила расчёта'
assert len(pandas_rows) == 4
print('Витрины идентичны:')
for row in pandas_rows:
    print(row)
print('Общая чистая сумма:', sum(row[3] for row in pandas_rows), 'руб.')


Витрины идентичны:
('email', 32800, 4300, 28500)
('loyalty', 37600, 3400, 34200)
('search', 34000, 5200, 28800)
('spring_ads', 34400, 2500, 31900)
Общая чистая сумма: 123400 руб.


**Контрольная точка 4.** Чем различаются два подхода? Какой из них относится к ETL, а какой — к ELT? Почему обе витрины совпали и какие ограничения есть у схемы `TRUNCATE` + загрузка? **Ваш ответ:** …
